In [1]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# estilo base
sns.set_theme(style="white")

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 14,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "lines.linewidth": 1,
    "axes.linewidth": 0.8,
    "figure.dpi": 300
})

# 1 coluna
FIGSIZE_SINGLE = (7, 5)

# 2 colunas
FIGSIZE_DOUBLE = (7, 3)

#palette = sns.color_palette("colorblind")
palette = sns.color_palette("Set2") #("flare") #("Set2") #("husl", 9) #("pastel")

# pasta com os ficheiros
DATA_DIR = "./"

data = []

def parse_filename(filename):
    # remove prefixo e extensão
    name = filename.replace("results_", "").replace(".json", "")
    
    parts = name.split("_")
    
    attack = parts[0]
    type = parts[1]  # adversarial, random, etc.
    feature = None
    restricted = False
    if len(parts) > 2:
        feature = parts[2]  # size, iat, both
        if len(parts) > 3:
            restricted = True

    return attack, type, feature, restricted


# ler todos os ficheiros
# iterar sobre as pastas na pasta
for folder in os.listdir(DATA_DIR):
    folder_path = os.path.join(DATA_DIR, folder)
    if os.path.isdir(folder_path) and folder in ["DDoS", "Mirai", "SSDP_Flood"]:
        for file in os.listdir(folder_path):
            if file.endswith(".json"):
                with open(os.path.join(folder_path, file)) as f:
                    content = json.load(f)
                
                    attack, type, feature, restricted = parse_filename(file)

                    if type == "original":
                        data.append({
                            "Attack": attack,
                            "Feature": type,
                            "Time Ratio": 1,
                            "Restricted": "Yes" if restricted else "No",
                            "Model": content["model"],
                            "ASR": content["metrics"]["attack_success_rate"]
                        })
                    else:
                        data.append({
                            "Attack": attack,
                            "Feature": feature,
                            "Time Ratio": content["Time_ratio"],
                            "Restricted": "Yes" if restricted else "No",
                            "Model": content["model"],
                            "ASR": content["metrics"]["attack_success_rate"]
                        })


df = pd.DataFrame(data)
print(df.head())
# garantir ordem consistente
feature_order = ["original", "iat", "size", "both"]
df["Feature"] = pd.Categorical(df["Feature"], categories=feature_order, ordered=True)

df["Feature"] = df["Feature"].map({
    "original": "Original",
    "iat": "IAT",
    "size": "Size",
    "both": "Both"
})

palette_fixed = {
    "Original": palette[0],
    "IAT": palette[1],
    "Size": palette[2],
    "Both": palette[3],
}

# garantir ordem consistente
attack_order = ["BruteForce", "DDoS", "PortScan", "Mirai", "SSDPFlood"]
df["Attack"] = pd.Categorical(df["Attack"], categories=attack_order, ordered=True)

df["Attack"] = df["Attack"].map({
    #"DDoS": "DDoS",
    "Mirai": "Mirai Botnet",
    "SSDPFlood": "SSDP Flood"
})

palette_fixed_attack = {
    "Mirai Botnet": palette[3],
    "SSDP Flood": palette[2],
}

  Attack   Feature  Time Ratio Restricted    Model       ASR
0  Mirai  original    1.000000         No  Kitsune  0.115561
1  Mirai       iat    7.250000        Yes  Kitsune  0.560000
2  Mirai      size    1.000000         No  Kitsune  0.115867
3  Mirai      both    1.000081         No  Kitsune  0.115867
4  Mirai       iat    8.378315         No  Kitsune  0.999953


In [2]:
for model in df["Model"].unique():
    # filtrar dados para o ataque atual
    subset = df[df["Model"] == model].copy()

    #filtrar apenas os dados sem restrição
    subset = subset[subset["Restricted"] == "No"]

    plt.figure(figsize=FIGSIZE_SINGLE)

    hue_order = ["Original", "IAT", "Size", "Both"]
    
    ax = sns.barplot(
        data=subset,
        x="Attack",
        y="ASR",
        hue="Feature",
        hue_order=hue_order,
        palette=palette_fixed,
        errorbar=None
    )

    ax.set_xlabel("Attack")
    ax.set_ylabel("ASR")

    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", fontsize=6, padding=2)

    # remover top/right spines
    sns.despine()

    # legenda compacta
    ax.legend(title="", frameon=False, loc="upper left", bbox_to_anchor=(1, 1))

    plt.tight_layout()
    plt.savefig(f"plots/ieee_asr_feature_{model}.png", bbox_inches="tight")
    plt.close()

In [3]:
for model in df["Attack"].unique():
    subset = df[df["Attack"] == model]

    subset = subset[subset["Feature"].isin(["Original", "IAT"])]

    plt.figure(figsize=FIGSIZE_SINGLE)

    ax = sns.lineplot(
        data=subset,
        x="Time Ratio",
        y="ASR",
        hue="Attack",
        marker="o",
        palette=palette,
        errorbar=None
    )

    ax.set_xlabel("Time Ratio")
    ax.set_ylabel("ASR")

    sns.despine()
    ax.legend(title="", frameon=False, loc="upper left", bbox_to_anchor=(1, 1))

    plt.tight_layout()
    plt.savefig(f"plots/ieee_asr_ratio_{model}.png", bbox_inches="tight")
    plt.close()

/tmp/user/1403/ipykernel_2120550/3870088498.py:8: UserWarning: The palette list has more values (8) than needed (1), which may not be intended.
  ax = sns.lineplot(
/tmp/user/1403/ipykernel_2120550/3870088498.py:8: UserWarning: The palette list has more values (8) than needed (1), which may not be intended.
  ax = sns.lineplot(


In [4]:
for model in df["Model"].unique():
    subset = df[df["Model"] == model]
    subset = subset[subset["Feature"].isin(["Original", "IAT"])]

    attacks = subset["Attack"].unique()
    fig, axes = plt.subplots(1, len(attacks), figsize=FIGSIZE_SINGLE, sharey=True)

    for ax, attack in zip(axes, attacks):
        data = subset[subset["Attack"] == attack]

        sns.lineplot(
            data=data,
            x="Time Ratio",
            y="ASR",
            hue="Attack",
            marker="o",
            palette=palette_fixed_attack,
            errorbar=None,
            ax=ax
        )

        #ax.set_title(attack)
        ax.set_xlabel("Time Ratio")
        ax.set_ylabel("ASR" if ax == axes[0] else "")
        ax.legend(title="", frameon=False)
        sns.despine(ax=ax)
         
    plt.tight_layout()
    plt.savefig(f"plots/ieee_asr_ratio_{model}.png", bbox_inches="tight")
    plt.close()

In [5]:
# Tables

table = df[["Attack", "Model", "Feature", "Time Ratio", "ASR"]]
table = table.sort_values(by=["Attack", "Model", "Feature", "Time Ratio"])
table.to_latex(
    "plots/results_table.tex",
    index=False,
    float_format="%.3f"
)

pivot = df.pivot_table(
    index=["Attack", "Feature", "Time Ratio"],
    columns="Model",
    values="ASR"
)

pivot.to_latex(
    "plots/asr_table.tex",
    float_format="%.3f"
)

/tmp/user/1403/ipykernel_2120550/625251239.py:11: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = df.pivot_table(


In [6]:
import numpy as np

for attack in df["Attack"].unique():
    subset = df[df["Attack"] == attack].copy()

    subset["Time Ratio"] = subset["Time Ratio"].round(2)

    # Limpar ratio para Original e Size
    subset["ratio_clean"] = np.where(
        subset["Feature"].isin(["Original", "Size"]),
        "",
        subset["Time Ratio"].astype(str)
    )

    # Ordenar ratios: primeiro vazio, depois os numéricos por ordem crescente
    ratios = sorted(
        subset[~subset["Feature"].isin(["Original", "Size"])]["Time Ratio"].unique()
    )
    ratio_order = [""] + [str(r) for r in ratios]
    subset["ratio_clean"] = pd.Categorical(subset["ratio_clean"], categories=ratio_order, ordered=True)

    # Pivot
    pivot = subset.pivot_table(
        index=["Model"],
        columns=["Feature", "ratio_clean"],
        values="ASR"
    )
    pivot = pivot.sort_index(axis=1, level=[0, 1])

    # Substituir NaN por traço
    pivot = pivot.fillna("—")

    pivot.to_latex(
        f"plots/asr_table_{attack}.tex",
        float_format="%.3f",
        multicolumn=True,
        multicolumn_format="c",
        na_rep="—"
    )

    print(f"Tabela gerada: plots/asr_table_{attack}.tex")

Tabela gerada: plots/asr_table_Mirai Botnet.tex
Tabela gerada: plots/asr_table_SSDP Flood.tex


/tmp/user/1403/ipykernel_2120550/3165521398.py:23: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = subset.pivot_table(
/tmp/user/1403/ipykernel_2120550/3165521398.py:23: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot = subset.pivot_table(


In [7]:
import numpy as np

df_table = df.copy()

df_table["Time Ratio"] = df_table["Time Ratio"].round(2)

# Criar label de linha: Feature + Ratio quando aplicável
def row_label(row):
    if row["Feature"] in ["Original", "Size"]:
        return row["Feature"]
    else:
        return f"IAT (ratio={row['Time Ratio']})"

df_table["Feature_Label"] = df_table.apply(row_label, axis=1)

# Ordenar linhas: Original, Size, IAT por ratio crescente
df_table["sort_key"] = df_table.apply(
    lambda r: (0 if r["Feature"] == "Original" else 1 if r["Feature"] == "Size" else 2, r["Time Ratio"]),
    axis=1
)
df_table = df_table.sort_values("sort_key")

# Pivot: linhas = Feature_Label, colunas = Attack
pivot = df_table.pivot_table(
    index="Feature_Label",
    columns="Attack",
    values="ASR",
    aggfunc="first"
)

# Garantir ordem das linhas
label_order = df_table.drop_duplicates("Feature_Label").sort_values("sort_key")["Feature_Label"].tolist()
pivot = pivot.loc[label_order]

pivot.to_latex(
    "plots/asr_table_full.tex",
    float_format="%.3f",
    multicolumn=True,
    multicolumn_format="c",
    na_rep="—"
)

In [8]:
import numpy as np

for attack in df["Attack"].unique():
    subset = df[df["Attack"] == attack].copy()

    subset["Time Ratio"] = subset["Time Ratio"].round(2)

    # Criar label de linha: Feature + Ratio quando aplicável
    def row_label(row):
        if row["Feature"] in ["Original", "Size"]:
            return row["Feature"]
        else:
            return f"IAT (ratio={row['Time Ratio']})"

    subset["Feature_Label"] = subset.apply(row_label, axis=1)

    # Ordenar linhas: Original, Size, IAT por ratio crescente
    subset["sort_key"] = subset.apply(
        lambda r: (0 if r["Feature"] == "Original" else 1 if r["Feature"] == "Size" else 2, r["Time Ratio"]),
        axis=1
    )
    subset = subset.sort_values("sort_key")

    # Pivot: linhas = Feature_Label, colunas = Attack
    pivot = subset.pivot_table(
        index="Feature_Label",
        columns="Attack",
        values="ASR",
        aggfunc="first"
    )

    # Garantir ordem das linhas
    label_order = subset.drop_duplicates("Feature_Label").sort_values("sort_key")["Feature_Label"].tolist()
    pivot = pivot.loc[label_order]

    pivot.to_latex(
        f"plots/asr_table_{attack}.tex",
        float_format="%.3f",
        multicolumn=True,
        multicolumn_format="c",
        na_rep="—"
    )

    print(f"Tabela gerada: plots/asr_table_{attack}.tex")

Tabela gerada: plots/asr_table_Mirai Botnet.tex
Tabela gerada: plots/asr_table_SSDP Flood.tex
